# B210: OCUDU gNB with a USRP B210 and a 5G modem

A 5G network on machines that are already provisioned: an Open5GS core, an OCUDU gNB driving a USRP B210, and a Quectel RM520N-GL modem (`qdon01`) as the UE.

- **Monitoring** (NADOR) collects RAN KPIs, N3 throughput per UE and slice, TCP round-trip time per UE and slice at the gNB and UPF, and the logs of the RAN, core and monitoring pods; the profile is [`monitoring.yaml`](monitoring.yaml).
- **Infrastructure:** Real radio hardware on `ptb-04015558` (gNB) and `pacify` (UE).

Run the cells in order; deployment uses real resources, and the Clean up section removes them. Monitoring is deployed after the RAN and before the UEs attach, so its sniffers see every session being set up.

## 1. Install

### Install IFRAN

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../../..").resolve()
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade nador
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


## 2. Describe the network

### Meta Model

In [ ]:
from metamodel_3gpp.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  b210:
    type: real
    txGainDB: 80
    rxGainDB: 40

ran:
  components:
    gnb:
      type: gnb
      config:
        radioConfiguration:
          nrBand: 78
          downlinkArfcn: 650000
          channelBandwidthMhz: 20
          prachConfigurationIndex: 1
          tddConfiguration:
            periodicityMilliseconds: 5
            nrofDownlinkSlots: 7
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 2
            nrofUplinkSymbols: 4
        ru: b210

ues:
  qdong01:
    type: cots
    config:
      supi: "001010000000017"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      subscriptions:
        - snssai: internet
          dnn: internet
      gnbs:
        - gnb
      nrBands:
        - 78
"""
)

### Deployment Model

In [ ]:
from ifran.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
nodes:
  ptb-04015558.inria.fr: # node's actual hostname
    is_provisioned: true
    provisioner: null

  pacify.inria.fr: # node's actual hostname
    is_provisioned: true
    provisioner: null
    ssh_info:
      user: ubuntu
      sudo_password: ubuntu

rus:
  b210:
    vendor: usrp
    model: b210
    is_provisioned: true
    provisioner: null

core:
  implementation: open5gs
  database_params:
    image:
      repository: "docker.io/mongo"
      image_tag: "7.0" # this is the mongodb version compatible with the machine's kernel version

ran:
  implementation: ocudu
  components:
    gnb:
      type: gnb
      advanced_config:
        srate: 23.04
"""
)

## 3. Kubernetes

In [ ]:
from ifran.deployment import K8SManagement

k8s = K8SManagement(metamodel=metamodel, deployment_model=deployment_model)

### Deploy Kubernetes only on `ptb-04015558.inria.fr` machine

In [ ]:
await k8s.deploy(nodes=["ptb-04015558.inria.fr"])

## 4. Core

In [ ]:
from ifran.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


### Check deployed pods with `kubectl`

In [ ]:
from ifran.deployment import ProvisioningManagement

prov = ProvisioningManagement(core)

print(
    (
        await prov.run_node_command(
            command=["kubectl", "get", "pods", "-n", "open5gs-core"],
            nodes=["ptb-04015558.inria.fr"],
            become=True,
            quiet=True,
            flatten=False,
        )
    )["ptb-04015558.inria.fr"]
)


## 5. RAN

### Check if USRP B210 is plugged in on `ptb-04015558.inria.fr`

In [ ]:
print(
    (await prov.run_node_command(command=["lsusb"], nodes=["ptb-04015558.inria.fr"], quiet=True, flatten=False))[
        "ptb-04015558.inria.fr"
    ]
)


Tune `ptb-04015558` (reference: https://kb.ettus.com/USRP_Host_Performance_Tuning_Tips_and_Tricks).

In [ ]:
print(
    (
        await prov.run_node_command(
            command=["apt", "install", "-y", "cpufrequtils"],
            nodes=["ptb-04015558.inria.fr"],
            become=True,
            quiet=True,
            flatten=False,
        )
    )["ptb-04015558.inria.fr"]
)


### Set the CPU governor to performance for all cores

In [ ]:
print(
    (
        await prov.run_node_command(
            command=["bash", "-c", "for ((i=0;i<$(nproc --all);i++)); do cpufreq-set -c $i -r -g performance; done"],
            nodes=["ptb-04015558.inria.fr"],
            become=True,
            quiet=True,
            flatten=False,
        )
    )["ptb-04015558.inria.fr"]
)


### Verify that the CPU governor has been set

In [ ]:
print(
    (
        await prov.run_node_command(
            command=["cpufreq-info"], nodes=["ptb-04015558.inria.fr"], become=True, quiet=True, flatten=False
        )
    )["ptb-04015558.inria.fr"]
)


### Deploy OCUDU gNB

In [ ]:
from ifran.deployment import RANManagement

ran = RANManagement(core)

from nador import MonitoringDeployment, configure_ran_kpis
ran_kpi_config = MonitoringDeployment.from_yaml_file("monitoring.yaml")
configure_ran_kpis(ran, ran_kpi_config)

In [ ]:
configure_ran_kpis(ran, ran_kpi_config)
await ran.deploy()

### Check gNB logs

In [ ]:
print(await ran.get_logs(quiet=True))

## 6. Monitoring (before the UEs attach)

The profile uses the control host selected by Kubernetes, including physical
Sopnodes. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from nador import MonitoringManagement
from nador import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )
prov = ProvisioningManagement(preceding=ran)

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

## 7. UEs

### Check if `qdong01` is plugged in on `pacify.inria.fr`

In [ ]:
print(
    (await prov.run_node_command(command=["lsusb"], nodes=["pacify.inria.fr"], quiet=True, flatten=False))[
        "pacify.inria.fr"
    ]
)


### Initialize UE

In [ ]:
print(
    (
        await prov.run_node_command(
            command=["qhat-init", "--dnn", "internet"],
            nodes=["pacify.inria.fr"],
            become=True,
            quiet=True,
            flatten=False,
        )
    )["pacify.inria.fr"]
)


> *Note*: If the command above fails with an error like this:
> ```bash
> [2026-10-05 04:21:40] 🔧 qhat-init — multi-modem init
> [2026-10-05 04:21:42] ❌ No modem AT port found.
> ```
> Then uncomment and run the `stop.sh` command and the `qhat-init` command again::

In [ ]:
# print((await prov.run_node_command(
#     command=["stop.sh"],
#     nodes=["pacify.inria.fr"],
#     become=True, quiet=True))["pacify.inria.fr"])

In [ ]:
# print((await prov.run_node_command(
#     command=["qhat-init", "--dnn", "internet"],
#     nodes=["pacify.inria.fr"],
#     become=True, quiet=True))["pacify.inria.fr"])

### Connect UE

In [ ]:
print(
    (
        await prov.run_node_command(
            command=["start.sh", "-F", "internet"], nodes=["pacify.inria.fr"], become=True, quiet=True, flatten=False
        )
    )["pacify.inria.fr"]
)


### Check if UE got a PDU session

In [ ]:
output = (
    await prov.run_node_command(
        command=["ip", "-4", "-o", "addr", "show", "wwan0"], nodes=["pacify.inria.fr"], quiet=True, flatten=False
    )
)["pacify.inria.fr"]

output


### Get UE IP

In [ ]:
ip = output.split()[3].split("/")[0]
ip

### Check gNB logs

In [ ]:
print(await ran.get_logs(quiet=True))

### Test connectivity

In [ ]:
print(
    (
        await prov.run_node_command(
            command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"], nodes=["pacify.inria.fr"], quiet=True, flatten=False
        )
    )["pacify.inria.fr"]
)


In [ ]:
print(
    (
        await prov.run_node_command(
            command=["ping", "-c", "6", "8.8.8.8", "-I", "wwan0"], nodes=["pacify.inria.fr"], quiet=True, flatten=False
        )
    )["pacify.inria.fr"]
)


## 8. Traffic and experiment

### Add necessary routing rules for the UE on `pacify.inria.fr` machine

In [ ]:
await prov.run_node_command(
    command=["ip", "route", "add", "11.1.1.0/29", "dev", "wwan0", "src", ip, "table", "100"],
    nodes=["pacify.inria.fr"],
    become=True,
    quiet=True,
    flatten=False,
)


In [ ]:
await prov.run_node_command(
    command=["ip", "route", "add", "default", "via", "11.1.1.1", "dev", "wwan0", "src", ip, "table", "100"],
    nodes=["pacify.inria.fr"],
    become=True,
    quiet=True,
    flatten=False,
)


In [ ]:
await prov.run_node_command(
    command=["ip", "rule", "add", "from", f"{ip}/32", "table", "100"],
    nodes=["pacify.inria.fr"],
    become=True,
    quiet=True,
    flatten=False,
)


### Iperf with experiment artifacts

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline.

In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun, iperf_summary
from ifran.deployment import ProvisioningManagement
from nador.traffic import Iperf

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = ["pacify.inria.fr"]
UE_INTERFACE = "wwan0"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

clients = ProvisioningManagement(preceding=ran)
iperf = await Iperf.setup(clients, server=SERVER_NODE, interface=UE_INTERFACE, clients=CLIENTS)

print(f"Server: {SERVER_NODE} ({iperf.server_ip}); UE clients: {iperf.ue_ips}")


`iperf.serve(port)` starts a one-shot iperf3 server on the server node. `iperf.client(ue, direction, seconds, port)` runs the UE side (`downlink` adds iperf's reverse mode) and returns iperf3's JSON report, with an `error` key when the test failed. `iperf.run(ue, direction, seconds)` does both on a fresh port.

In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'pacify.inria.fr', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'pacify.inria.fr', "direction": "downlink", "duration": DURATION},
]

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        await iperf.serve(port)

        with experiment.section(test["name"]):
            report = await iperf.client(
                client=test["client"],
                direction=test["direction"],
                duration=test["duration"],
                port=port,
            )

        experiment.save_json(f"{test['name']}.json", report)
        if report.get("error"):
            raise RuntimeError(f"{test['name']}: {report['error']}")

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")
print(iperf_summary(output_directory))

### Save meta model and deployment model

In [ ]:
ran.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

## 9. Clean up

### Disconnect UE

In [ ]:
# from ifran.deployment import ProvisioningManagement

# prov = ProvisioningManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# print((await prov.run_node_command(
#     command=["stop.sh"],
#     nodes=["pacify.inria.fr"],
#     become=True, quiet=True))["pacify.inria.fr"])

### Remove RAN

In [ ]:
# from ifran.deployment import RANManagement

# ran = RANManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await ran.remove()

### Remove 5G core

In [ ]:
# from ifran.deployment import CoreManagement

# core = CoreManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await core.remove()

### Remove Kubernetes

In [ ]:
# from ifran.deployment import K8SManagement

# k8s = K8SManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await k8s.remove()